In [1]:
# Project Goal:
# Combining electronics with machine learning.
# This is an object classifier — I took all the training photos myself.
# Goal: connect this to Arduino so that when the webcam sees an object,
# it prints the object's name on an OLED display,
# and a servo motor moves to a specific angle depending on the object detected.

import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
# Configuration

img_size = (128, 128)      # resize all images to 128x128 before feeding into the model
batch_size = 16            # number of images processed together in one training step
epochs = 20                # max number of times the model will see the full dataset

train_dir = r"C:\Users\KOKO\Desktop\objects\datasets\train"
test_dir = r"C:\Users\KOKO\Desktop\objects\datasets\test"

In [3]:
# Load datasets

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
print("Classes found:", class_names)

Found 679 files belonging to 5 classes.
Found 237 files belonging to 5 classes.
Classes found: ['apple', 'banana', 'cup', 'juice', 'phone']


In [4]:
# Performance optimization

autotune = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=autotune)
test_ds = test_ds.cache().prefetch(buffer_size=autotune)

In [5]:
# Data augmentation
# Since the dataset is homemade, augmentation helps the model
# generalize better to lighting/angle variations it hasn't seen.

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
])

In [6]:
# Build the CNN

num_classes = len(class_names)

model = models.Sequential([
    layers.Input(shape=(img_size[0], img_size[1], 3)),
    layers.Rescaling(1./255),
    data_augmentation,

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.4),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,285 (12.61 MB)

 Trainable params: 3,305,285 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Compile the model

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [8]:
# Early stopping
# Stops training if validation accuracy stops improving,
# so we don't waste time or overfit by training too long.early_stop = tf.keras.callbacks.EarlyStopping(

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True
)

# Train the model

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=epochs,
    callbacks=[early_stop]
)

Epoch 1/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 92ms/step - accuracy: 0.2563 - loss: 1.5815 - val_accuracy: 0.4473 - val_loss: 1.3082
Epoch 2/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.5214 - loss: 1.1308 - val_accuracy: 0.7215 - val_loss: 0.6900
Epoch 3/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 0.6524 - loss: 0.8065 - val_accuracy: 0.7342 - val_loss: 0.7701
Epoch 4/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 0.6848 - loss: 0.7529 - val_accuracy: 0.7131 - val_loss: 0.6890
Epoch 5/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 0.7437 - loss: 0.6549 - val_accuracy: 0.7722 - val_loss: 0.7133
Epoch 6/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 0.7614 - loss: 0.6013 - val_accuracy: 0.8312 - val_loss: 0.5513
Epoch 7/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - accuracy: 0.8586 - loss: 0.3921 - val_accuracy: 0.8861 - val_loss: 0.3788
Epoch 8/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - accuracy: 0.8704 - loss: 0.3428 - val_accuracy: 0.8861 - v

In [9]:
# Evaluate on test set

test_loss, test_acc = model.evaluate(test_ds)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")

# Save the model

model.save("object_classifier.keras")
print("Model saved as object_classifier.keras")

# Save class names too — you'll need this exact order later
# for the webcam/Arduino inference script

with open("class_names.txt", "w") as f:
    f.write("\n".join(class_names))

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9662 - loss: 0.2413

Final Test Accuracy: 96.62%
Model saved as object_classifier.keras


In [10]:
# Real-time object detection using webcam

import cv2
import numpy as np
import tensorflow as tf

# Load the trained model
model = tf.keras.models.load_model("object_classifier.keras")

# Load class names (must match training order)
with open("class_names.txt", "r") as f:
    class_names = f.read().splitlines()

print("Loaded classes:", class_names)

img_size = (128, 128)

Loaded classes: ['apple', 'banana', 'cup', 'juice', 'phone']


In [12]:
import serial
import time

arduino = serial.Serial('COM7', 9600, timeout=1)
time.sleep(2)

In [15]:
cap = cv2.VideoCapture(1)

last_sent = None  # NEW: track last sent class, so we don't spam Arduino every single frame

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    img = cv2.resize(frame, img_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.expand_dims(img, axis=0)

    predictions = model.predict(img, verbose=0)
    class_index = np.argmax(predictions[0])
    class_name = class_names[class_index]
    confidence = predictions[0][class_index] * 100

    # NEW: only send to Arduino if the prediction changed AND confidence is high enough
    if class_name != last_sent and confidence > 80:
        arduino.write((class_name + "\n").encode())
        print(f"Sent to Arduino: {class_name}")
        last_sent = class_name

    label = f"{class_name} ({confidence:.1f}%)"
    cv2.putText(frame, label, (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.imshow("Object Classifier", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
arduino.close()  # NEW: cleanly close serial connection when done
   

PortNotOpenError: Attempting to use a port that is not open

In [ ]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()
for port in ports:
    print(port.device, port.description)

COM7 USB Serial Device (COM7)
